# Multi-View Building Discovery, Facade Analysis & Cost Tracking

[Open in Google Colab][colab] · [Open in Colab Enterprise][colab-ent] · [View on GitHub][github]

Property insurance underwriting, municipal tax assessment, and solar/retrofit screening workflows
typically begin with approximate parcel coordinates `(LAT, LNG)` and need unoccluded, well-framed
street-level imagery of the target building. This tutorial discovers nearby 7-camera panoramic
rosettes (`capture_id`) in BigQuery, ranks camera angles that frame the target building with
`< 1%` analytic black border, extracts structured architectural attributes (`stories`,
`exterior_material`, `roof_type`) with Vertex AI Gemini, verifies storey counts in Gemini's Python
code-execution sandbox, and triangulates the physical facade position from 2+ intersecting camera
rays.

| Stage | Tool | Responsibility |
|---|---|---|
| **1. Discovery** | BigQuery (`rosette_sql`) | Find rosettes (`<= 80 m`) and target bearings |
| **2. Ranking** | `svi_geo.views` | Rank cameras (`cam_0..cam_5`) across drives (`< 1%` black) |
| **3. Perception** | Gemini (`HouseView`) | Extract visibility, stories, exterior, and roof |
| **4. Verify** | Code Exec + 3D Rays | Count window rows in sandbox and triangulate 3D facade |

> **Runtime & Cost Banner:** Expected runtime **<= 6 min** on Colab CPU · Gemini ceiling
> **<= $0.15** (`<= 12` plain calls + `1` code-exec call) · Cold BigQuery scan **1.83 GB** at
> **$6.25/TB** (~$0.011 first run, **0 B** on cached re-runs within 35 days).

[colab]: https://colab.research.google.com/github/googlemaps-samples/insights-samples
[colab-ent]: https://console.cloud.google.com/vertex-ai/colab/import
[github]: https://github.com/googlemaps-samples/insights-samples/tree/main/street_view_insights

## 1. Architecture, Data Model & Division of Labour

- **Problem Statement**: Given approximate `(LAT, LNG)` coordinates of a building, select
  non-occluded views across nearby drive sequences, extract and fuse architectural facade
  attributes (`stories`, `exterior_material`, `roof_type`), and triangulate the physical facade
  location from 2+ camera sighting rays. Reported self-consistency metrics measure multi-view
  agreement, not ground-truth accuracy.
- **Data Model & First-Run BigQuery Cost**: Every panoramic capture of 7 frames (`_0`..`_6`) is
  keyed by `capture_id`; `pano_id` is nullable attribution metadata (`~69%` of rosettes in typical
  neighbourhoods have `pano_id IS NULL`). A cold scan of `rosette_sql` bills **1.83 GB** at
  **$6.25/TB** on-demand (cached locally up to 35 days so warm re-runs bill **0 B**).
- **Division of labour**:
  - **BigQuery**: Computes drive sequences, camera-to-target bearings, and radial distances.
  - **Python (`svi_geo`)**: Handles Kannala–Brandt (`KB4`) fisheye perspective projection,
    sensor-coverage view ranking, multi-view attribute fusion, and 3D facade ray triangulation.
  - **Vertex AI Gemini**: Predicts structured `HouseView` attributes across candidate views and
    executes one **agentic** vision (`code_execution=True`) check to count horizontal facade
    window rows in the Python sandbox, cross-checked against local Sobel-y gradient peaks.

## 2. Terms of Service, Attribution & Authentication

**Terms of use & attribution.** Imagery comes from your Street View Insights dataset and is
governed by the terms of your Google Cloud agreement: do not redistribute frames or rendered crops
outside your organisation, keep the `"Imagery © Google"` attribution on every figure and map, and
do not persist raw imagery or `gcs_uri` values beyond short-lived working caches (`<= 24 h` for
frames, `<= 35 days` for snapshot-keyed metadata).

- **Colab:** calls `google.colab.auth.authenticate_user()`.
- **Local Jupyter:** run `gcloud auth application-default login` once.
- **Workstation helpers:** set `SVI_USE_GCLOUD_TOKEN=1` and `SVI_ECP_PROXY_URL` when required.
- **IAM roles needed:** `roles/bigquery.jobUser`, `roles/bigquery.dataViewer`,
  `roles/storage.objectViewer`, and `roles/aiplatform.user`.

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# @title Step 1: Install the svi_geo Helper Package
# Install the shared `svi_geo` helper package (geometry, sequencing, dedup, Gemini client).
# Dependency ranges in pyproject.toml are compatible with Colab preinstalled versions.
# In a clone of the repo the local copy is used: $SVI_GEO_LOCAL_PATH, or the first parent
# directory that contains street_view_insights/panoramic/svi_geo. Otherwise pip installs it
# from GitHub at SVI_GEO_REF.
import importlib.util
import os
import pathlib
import subprocess
import sys

# TODO(after-merge): set to "main" once svi_geo is merged. Override with $SVI_GEO_REF.
SVI_GEO_REF = os.environ.get("SVI_GEO_REF", "f4087e9")


def find_local_svi_geo():
    if os.environ.get("SVI_GEO_LOCAL_PATH"):
        return os.environ["SVI_GEO_LOCAL_PATH"]
    here = pathlib.Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "street_view_insights/panoramic/svi_geo" / "pyproject.toml").is_file():
            return str(d / "street_view_insights/panoramic/svi_geo")
    return None


LOCAL_SVI_GEO = find_local_svi_geo()
if importlib.util.find_spec("svi_geo") is None:
    if LOCAL_SVI_GEO:
        src = f"{LOCAL_SVI_GEO}[notebooks]"
    else:
        src = (
            "svi-geo[notebooks] @ git+https://github.com/googlemaps-samples/insights-samples"
            f"@{SVI_GEO_REF}#subdirectory=street_view_insights/panoramic/svi_geo"
        )
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", src])

## 3. Environment Configuration & Target Building Parameters

Here we configure the Google Cloud environment (`PROJECT_ID`, `GCS_BUCKET`), the target building
coordinates (`LAT`, `LNG`, `RADIUS_M = 80.0 m`), view diversity caps (`MAX_VIEWS = 8`,
`MAX_PER_SEQ = 4`, `VIEW_W = 1024`, `VIEW_H = 768`), and Vertex AI Gemini budget guardrails
(`MODEL = gc.DEFAULT_MODEL`, `THINKING_LEVEL = "MEDIUM"`, `MEDIA_RESOLUTION = "MEDIUM"`,
`MAX_GEMINI_CALLS = 25`, `MAX_USD = 0.15`).

In [ ]:
# @title Step 2: Configure Parameters & Initialize Cloud Clients
import json
import os

import matplotlib.pyplot as plt
from google.cloud import storage
from IPython.display import Image, display

from svi_geo import attribution, auth, config, data, images, maps, rosette, schemas, usecases, views
from svi_geo import gemini_client as gc

try:
    from google.colab import auth as colab_auth

    colab_auth.authenticate_user()
except ImportError:
    pass

PROJECT_ID = os.environ.get("PROJECT_ID", "YOUR_PROJECT_ID")
DATASET_ID = "imagery_insights___us"
GCS_BUCKET = os.environ.get("GCS_BUCKET", "")
SNAPSHOT_ID = None
INCLUDE_UNPUBLISHED_PANOS = True
MODEL = gc.DEFAULT_MODEL
LOCATION = "global"
THINKING_LEVEL = "MEDIUM"
MEDIA_RESOLUTION = "MEDIUM"
ENABLE_CODE_EXECUTION = True
MAX_GEMINI_CALLS = 25
MAX_USD = 0.15
CONCURRENCY = 8
SEED = 0
LAT, LNG = 28.05047, -81.96015
RADIUS_M = 80.0
MAX_VIEWS = 8
MAX_PER_SEQ = 4
VIEW_W, VIEW_H = 1024, 768
OUT_DIR = pathlib.Path(os.environ.get("OUT_DIR", "~/tmp/svi_qa/uc1_house")).expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)
QA_DIR = (
    pathlib.Path(LOCAL_SVI_GEO or ".") / "data" / "qa" / "2026-10-01" if LOCAL_SVI_GEO else OUT_DIR
)

settings = config.resolve_settings(PROJECT_ID, GCS_BUCKET, env=dict(os.environ), dataset=DATASET_ID)
PROJECT_ID, BUCKET = settings.project, settings.bucket
creds = auth.get_credentials()
TABLE = data.pano_table(PROJECT_ID, DATASET_ID)
SNAP_TABLE = f"{PROJECT_ID}.{DATASET_ID}.snapshots"
bq = data.QueryRunner(
    data.make_bigquery_client(PROJECT_ID, creds),
    allowed_tables=data.pano_tables(PROJECT_ID, DATASET_ID),
    cache_dir=data.DEFAULT_QUERY_CACHE,
)
fetcher = images.GcsImageFetcher(storage.Client(project=PROJECT_ID, credentials=creds))
runner = gc.GeminiRunner(
    gc.VertexGeminiBackend(
        gc.make_vertex_client(PROJECT_ID, LOCATION, creds),
        model=MODEL,
        thinking_level=THINKING_LEVEL,
        media_resolution=MEDIA_RESOLUTION,
    ),
    max_calls=MAX_GEMINI_CALLS,
    max_usd=MAX_USD,
    concurrency=CONCURRENCY,
)
intr = rosette.load_intrinsics()

## 4. Discover Nearby Panoramic Rosettes & Attach Target Bearings in BigQuery

We first query `snapshot_catalog_sql` (`2 KB`) to inspect live snapshots and then run `rosette_sql`
within `RADIUS_M = 80.0 m` of `(LAT, LNG)`. Calling `data.attach_target_framing` enriches each
rosette with the distance (`dist_m`) and world bearing (`target_bearing_deg`) from the capture
position to the target building coordinates.

In [ ]:
# @title Step 3: Query Nearby Rosettes & Attach Target Bearings
snap_df = bq.run(data.snapshot_catalog_sql(SNAP_TABLE), [])
ROSETTE_SQL = data.rosette_sql(TABLE)
print(ROSETTE_SQL[:360] + "\n...")
params = data.rosette_params(
    lat=LAT,
    lng=LNG,
    radius_m=RADIUS_M,
    snapshot_id=SNAPSHOT_ID,
    include_unpublished=INCLUDE_UNPUBLISHED_PANOS,
)
rosettes = data.attach_target_framing(
    bq.run(ROSETTE_SQL, params, template_name="rosette_sql"), tlat=LAT, tlng=LNG
)
snap_info = data.summarize_snapshots(rosettes, snap_df)
print(snap_info["summary_line"])
frames = data.frames_from_rosettes(rosettes, bucket=BUCKET)
cost_bq = bq.last_cost()
snap_ids = ",".join(s[:8] for s in snap_info["snapshot_ids"]) or "none"
null_cnt = int(rosettes["pano_id"].isna().sum())
print(
    f"snapshots={snap_ids} rosettes={len(rosettes)} null_pano_id={null_cnt} "
    f"sequences={rosettes['seq_id'].nunique()} bq_bytes={cost_bq['bytes']} "
    f"bq_cached={int(cost_bq['cached'])}"
)

## 5. Rank Candidate Camera Views & Verify Sensor Coverage

Each rosette has 6 horizontal cameras (`cam_k = 0..5`). `views.house_view_candidates` selects the
camera whose optical axis points closest to the target building and sizes an adaptive horizontal
field of view (`hfov_deg`) based on distance while guaranteeing via `rosette.view_black_fraction`
that the rendered perspective crop has `< 1%` analytic sensor border miss. `views.rank_house_views`
then selects up to `MAX_VIEWS = 8` views spread across distinct drive sequences (`max_per_seq=4`)
to maximize stereo baseline diversity.

In [ ]:
# @title Step 4: Rank Candidate Building Views & Check Sensor Coverage
VARIANT = usecases.DEFAULT_VARIANT
print(VARIANT.describe())
cands = views.house_view_candidates(frames, LAT, LNG, intr, aspect=VIEW_W / VIEW_H)
ranked = views.rank_house_views(cands, max_per_seq=MAX_PER_SEQ, n=MAX_VIEWS)
if ranked.empty:
    raise RuntimeError("No candidate house views found within RADIUS_M.")
r0 = ranked.iloc[0].to_dict()
v0 = views.view_for(r0, VIEW_W, VIEW_H)
b0 = rosette.view_black_fraction(intr, r0["camera_pose"], v0, int(r0["cam_k"]))
az0, el0 = v0.pixel_to_bearing(VIEW_W / 2, VIEW_H / 2)
print(
    f"worked_view capture_id={r0['capture_id']} cam_k={r0['cam_k']} "
    f"hfov={v0.hfov_deg:.1f} black={b0:.4f} ray=({az0:.1f},{el0:.1f})"
)

## 6. Structured Architectural Attribute Extraction with Gemini

We inspect the prompt (`usecases.UC1_PROMPT_V0`) and Pydantic schema (`schemas.HouseView`:
`visible`, `centred`, `occlusion`, `box_2d`, `stories`, `exterior_material`, `roof_type`,
`confidence`), run a single-view demonstration on the top-ranked perspective crop (`crop0`), and
then execute `usecases.uc1_run` across all ranked candidate views to fuse attributes weighted by
confidence, framing non-truncation, and low privacy-mask overlap.

In [ ]:
# @title Step 5: Extract Structured Building Attributes Across Ranked Views
print("Prompt:", usecases.UC1_PROMPT_V0)
print("Schema keys:", list(schemas.HouseView.model_json_schema()["properties"]))
raw0 = images.decode(
    fetcher.fetch(r0["gcs_uri"]), scale=images.decode_scale_for_view(VIEW_W, v0.hfov_deg)
)
crop0 = rosette.render_perspective(
    raw0, intr, r0["camera_pose"], v0, int(r0["cam_k"]), antialias=True, mask_hood=True
)
demo_hv = await runner.ask([usecases.UC1_PROMPT_V0, crop0], schemas.HouseView, seed=SEED)
print("Single-view demo:", demo_hv.model_dump())
out = await usecases.uc1_run(
    ranked, fetcher.fetch, runner, intr, width=VIEW_W, height=VIEW_H, variant=VARIANT
)
runner.check(context="UC1 HouseView")
res = out["res"]
assert len(res) > 0 and int(res["centred"].sum()) >= 1

## 7. Agentic Vision Verification: Counting Facade Storeys in the Sandbox

To cross-check the multi-view fused `stories` attribute with image-level structural evidence, we
run one **agentic** vision call (`usecases.uc1_count_storeys` with `code_execution=True`). Inside
Gemini's secure Python sandbox, the model crops the building facade, computes horizontal Sobel-y
gradient profiles to count window/storey rows, and emits a structured measurement that we validate
with `gc.check_code_exec_trace` and compare against our fused storey estimate
(`code_exec_agreement=agree(...)`).

In [ ]:
# @title Step 6: Verify Storey Count with Gemini Code Execution
centred_idx = [i for i, c in enumerate(out["res"]["centred"]) if c]
best_i = centred_idx[0] if centred_idx else 0
best_view = out["views_out"][best_i]
fused_stories = out["attrs"]["stories"][0]
ce = await usecases.uc1_count_storeys(
    out["crops"][best_i],
    runner,
    box_2d=best_view.box_2d if best_view else None,
    fused_stories=int(fused_stories) if fused_stories else None,
    seed=SEED,
    thinking_level=THINKING_LEVEL,
    code_execution=True,
    validator=lambda r: None,
    expect_stdout=usecases.MEASURE_STDOUT_PATTERN,
)
gc.check_code_exec_trace(ce["trace"], expect_stdout=usecases.MEASURE_STDOUT_PATTERN)
assert ce["agreement_line"].startswith("code_exec_agreement=")
print(ce["agreement_line"])
ce_path = OUT_DIR / "agentic_storeys.png"
fig_ce, ax_ce = plt.subplots(figsize=(5, 4))
ax_ce.imshow(ce["overlay"][:, :, ::-1])
ax_ce.set_title(f"Agentic Storey Check: {ce['agreement_line']}")
ax_ce.axis("off")
attribution.save_figure(fig_ce, ce_path)
plt.close(fig_ce)
display(Image(filename=str(ce_path)))

## 8. Multi-View 3D Facade Triangulation & Quality Gates

Each centred `HouseView` bounding box yields a calibrated 3D world sighting ray from its camera
origin. When 2 or more rays intersect within geometric bounds, `usecases.uc1_run` triangulates the
building facade coordinates `(loc.lat, loc.lng)` in local East-North-Up (`ENU`) metres and reports
the root-mean-square ray residual (`loc.rms_m`), alongside sensor coverage (`black_fraction_max`),
dark-pixel fraction (`dark_pixel_max`), and privacy-redaction overlap (`redaction_overlap_max`).

In [ ]:
# @title Step 7: Inspect Triangulated Coordinates, Ray Residual & Quality Gates
loc = out["location"]
print(
    f"black_fraction_max={out['black_fraction_max']:.4f} "
    f"dark_pixel_max={out['dark_pixel_max']:.4f} "
    f"redaction_overlap_max={out['redaction_overlap_max']:.4f}"
)
print(f"centred_views={int(res['centred'].sum())} status={out['status']} attrs={out['attrs']}")
if loc is not None:
    print(f"triangulated=({loc.lat:.6f}, {loc.lng:.6f}) rms_m={loc.rms_m:.2f}")

## 9. Sighting-Ray Triangulation Map & QA Overlay Grid

We now render two visual deliverables:

1. **Interactive Sighting-Ray Map (`map.html`)**: Plots the nearby rosette positions, camera-to-
   facade sighting rays, the input target pin, and the triangulated facade location with mandatory
   `"Imagery © Google"` attribution.
2. **QA Overlay Grid (`overlay_grid.png`)**: Displays the top candidate perspective crops, drawing
   accepted `centred_house` boxes in green and rejected views (`truncated`, `occluded`, or
   `off_centre`) in dashed red.

In [ ]:
# @title Step 8: Render Sighting-Ray Map & QA Overlay Grid
m = maps.uc1_house_map(
    rosettes,
    target_lat=LAT,
    target_lng=LNG,
    sightings=out["sightings"],
    location=out["location"],
    out_path=OUT_DIR / "map.html",
)
panels = []
for idx in range(min(3, len(out["crops"]))):
    v = out["views_out"][idx]
    row_i = res.iloc[idx]
    box_px = (
        schemas.box_2d_to_pixels(v.box_2d, VIEW_W, VIEW_H)
        if (v and v.box_2d)
        else (80, 80, VIEW_W - 80, VIEW_H - 80)
    )
    is_ok = bool(row_i["centred"])
    is_trunc = bool(row_i.get("truncated", False))
    is_occ = bool(v and getattr(v, "occlusion", "NONE") not in ("NONE", None))
    reason = "truncated" if is_trunc else ("occluded" if is_occ else "off_centre")
    panels.append(
        {
            "title": f"view_{idx} ({row_i['dist_m']:.0f}m)",
            "image": out["crops"][idx],
            "accepted_boxes": [(*box_px, "centred_house")] if is_ok else [],
            "rejected_boxes": [] if is_ok else [(*box_px, reason)],
        }
    )
grid_path = OUT_DIR / "overlay_grid.png"
maps.render_overlay_grid(
    panels,
    grid_path,
    notebook_name="house_image_discovery_with_cost",
    qa_dir=QA_DIR,
)
display(Image(filename=str(grid_path)))

## 10. Token Scaling Preview & Cost Audit

Before scaling building attribute extraction to thousands of parcels, `gc.preview_tokens` previews
multimodal input token counts across `LOW`, `MEDIUM`, and `HIGH` media resolutions, and
`runner.cost.summary` verifies that actual Gemini API spend stayed within `MAX_USD = $0.15`.

In [ ]:
# @title Step 9: Preview Token Scaling & Export Cost Telemetry
prev_by_res = {
    lvl: gc.preview_tokens(
        runner.backend, [usecases.UC1_PROMPT_V0, out["crops"][0]], media_resolution=lvl
    )
    for lvl in ("LOW", "MEDIUM", "HIGH")
}
est = gc.estimate_cost(len(out["crops"]), preview=prev_by_res[MEDIA_RESOLUTION])
print(f"preview_tokens={prev_by_res} pre_run_est_usd=${est:.4f} ceiling=${MAX_USD:.2f}")
print(runner.cost.summary(ceiling=MAX_USD))
(OUT_DIR / "metrics.json").write_text(
    json.dumps(
        {
            "notebook": "house_image_discovery_with_cost",
            "rosettes": int(len(rosettes)),
            "null_pano_id": int(rosettes["pano_id"].isna().sum()),
            "centred_views": int(res["centred"].sum()),
            "est_cost_usd": float(runner.cost.usd),
        },
        indent=2,
    ),
    encoding="utf-8",
)
artefacts = len(list(OUT_DIR.glob("*")))
print(f"map_rendered=1 artefacts={artefacts} peak_rss_mb={images.peak_rss_mb():.0f}")

## 11. Summary, Generated Artifacts & Next Steps

### Summary & Generated Artifacts

- **Pipeline Recap**: You queried nearby `capture_id` rosettes in BigQuery, ranked unoccluded
  building views with `< 1%` analytic black border, fused `HouseView` attributes across views,
  verified storey counts in the Gemini code-execution sandbox, and triangulated the building facade
  from intersecting 3D rays.
- **Saved Artifacts in `OUT_DIR`**: `map.html` (sighting-ray & triangulated facade map),
  `overlay_grid.png` (accepted vs rejected candidate views), `agentic_storeys.png` (Sobel-y
  storey-count verification), and `metrics.json`.
- **Operational Guidance**: Views with heavy tree occlusion or privacy redaction masks
  (`redaction_overlap > 0.30`) are automatically down-weighted during multi-view attribute fusion.

### Next Steps

- Explore **[Sequential Asset & Urban Forestry Detection](analyze_sequential_images.ipynb)**
  for corridor-scale public right-of-way infrastructure and tree inventory.
- Explore **[Roof Edge Polyline Tracing](roof_edge_tracing.ipynb)** for high-resolution roof ridge
  and eave geometry extraction, and see `svi_geo/EVALUATION.md` for multi-AOI consistency results.